In [1]:
# Import core libraries
import pandas as pd
import numpy as np
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, Descriptors, Draw, PandasTools, Lipinski, Crippen
from rdkit.Chem.FilterCatalog import FilterCatalog, FilterCatalogParams
from rdkit.Chem.Pharm2D import Generate, Gobbi_Pharm2D
from rdkit.ML.Descriptors import MoleculeDescriptors
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend
import seaborn as sns
import requests
import py3Dmol
#from openbabel import pybel
import os

print("All libraries imported successfully!")

<frozen importlib._bootstrap>:488: RuntimeWarning: to-Python converter for boost::shared_ptr<RDKit::FilterHierarchyMatcher> already registered; second conversion method ignored.


All libraries imported successfully!


In [2]:
# 14.2 Drug Discovery Process

# Fetch and visualize HER2 structure (PDB: 3PP0)
def fetch_pdb_structure(pdb_id):
    """
    Fetch PDB structure from RCSB database
    Args:
        pdb_id (str): PDB ID of the structure to fetch
    Returns:
        str: Path to the downloaded PDB file or None if failed
    """
    # Fix URL format by removing extra spaces
    url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
    try:
        response = requests.get(url)
        response.raise_for_status()  # Raise an exception for bad status codes
        with open(f"{pdb_id}.pdb", "w") as f:
            f.write(response.text)
        return f"{pdb_id}.pdb"
    except Exception as e:
        print(f"Error fetching PDB structure: {e}")
        return None

# Download HER2 structure
pdb_file = fetch_pdb_structure("3PP0")

# Visualize the structure
def visualize_pdb(pdb_file):
    """
    Visualize PDB structure using py3Dmol
    Args:
        pdb_file (str): Path to the PDB file
    Returns:
        py3Dmol.view: Visualization object or None if failed
    """
    if not pdb_file or not os.path.exists(pdb_file):
        print("PDB file not found")
        return None

    try:
        with open(pdb_file, "r") as f:
            pdb_data = f.read()

        view = py3Dmol.view(width=400, height=300)
        view.addModel(pdb_data, "pdb")
        view.setStyle({'cartoon': {'color': 'spectrum'}})
        view.zoomTo()
        # Since we're using Agg backend, return view object instead of calling show()
        return view
    except Exception as e:
        print(f"Error visualizing PDB: {e}")
        return None

if pdb_file:
    view = visualize_pdb(pdb_file)
    if view:
        # Save visualization as HTML file
        with open("protein_view.html", "w") as f:
            f.write(view._make_html())


In [3]:
# 14.3 Data Sources for Drugs, Targets, and Diseases

# Fetch HER2 inhibitors from ChEMBL
def fetch_her2_inhibitors():
    """
    Fetch HER2 inhibitors data from ChEMBL database
    Returns:
        pandas.DataFrame: DataFrame containing molecule information
    """
    # Fix URL format by removing extra spaces
    url = "https://www.ebi.ac.uk/chembl/api/data/molecule.json?target_chembl_id=CHEMBL1941&max_results=50"
    try:
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        molecules = []
        for compound in data['molecules']:
            if 'molecule_structures' in compound and 'canonical_smiles' in compound['molecule_structures']:
                molecules.append({
                    'chembl_id': compound['molecule_chembl_id'],
                    'smiles': compound['molecule_structures']['canonical_smiles'],
                    'name': compound['pref_name'] if compound['pref_name'] else 'Unknown'
                })

        return pd.DataFrame(molecules)
    except Exception as e:
        print(f"Error fetching HER2 inhibitors: {e}")
        return pd.DataFrame()

# Load HER2 inhibitors
her2_df = fetch_her2_inhibitors()
print(f"Retrieved {len(her2_df)} HER2 inhibitors")
print(her2_df.head())

Retrieved 20 HER2 inhibitors
      chembl_id                                            smiles     name
0    CHEMBL6329      Cc1cc(-n2ncc(=O)[nH]c2=O)ccc1C(=O)c1ccccc1Cl  Unknown
1    CHEMBL6328   Cc1cc(-n2ncc(=O)[nH]c2=O)ccc1C(=O)c1ccc(C#N)cc1  Unknown
2  CHEMBL265667  Cc1cc(-n2ncc(=O)[nH]c2=O)cc(C)c1C(O)c1ccc(Cl)cc1  Unknown
3    CHEMBL6362      Cc1ccc(C(=O)c2ccc(-n3ncc(=O)[nH]c3=O)cc2)cc1  Unknown
4  CHEMBL267864    Cc1cc(-n2ncc(=O)[nH]c2=O)ccc1C(=O)c1ccc(Cl)cc1  Unknown


In [4]:
# 14.4 Druglikeness

# Calculate druglikeness properties
def calculate_drug_properties(smiles):
    """
    Calculate drug-likeness properties for a molecule
    Args:
        smiles (str): SMILES representation of the molecule
    Returns:
        dict: Dictionary containing molecular properties or None if invalid SMILES
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    return {
        'Molecular_Weight': Descriptors.MolWt(mol),
        'LogP': Crippen.MolLogP(mol),
        'H_Bond_Donors': Lipinski.NumHDonors(mol),
        'H_Bond_Acceptors': Lipinski.NumHAcceptors(mol),
        'Rotatable_Bonds': Lipinski.NumRotatableBonds(mol),
        'Passes_Lipinski': Lipinski.NumHAcceptors(mol) >= 1 and Lipinski.NumHDonors(mol) >= 1 and
                           Descriptors.MolWt(mol) <= 500 and Crippen.MolLogP(mol) <= 5
    }

# Add properties to dataframe
her2_df['properties'] = her2_df['smiles'].apply(calculate_drug_properties)

# Expand properties into separate columns
properties_df = her2_df['properties'].apply(pd.Series)
her2_df = pd.concat([her2_df.drop('properties', axis=1), properties_df], axis=1)

print("Drug properties calculated:")
print(her2_df.head())

# Plot molecular weight distribution
if 'Molecular_Weight' in her2_df.columns:
    plt.figure(figsize=(10, 6))
    sns.histplot(her2_df['Molecular_Weight'].dropna(), bins=20, kde=True)
    plt.axvline(500, color='r', linestyle='--', label='MW ≤ 500')
    plt.title('Molecular Weight Distribution of HER2 Inhibitors')
    plt.xlabel('Molecular Weight')
    plt.legend()
    # Use savefig instead of show() for non-interactive backend
    plt.savefig('molecular_weight_distribution.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("Molecular weight distribution plot saved as 'molecular_weight_distribution.png'")


Drug properties calculated:
      chembl_id                                            smiles     name  \
0    CHEMBL6329      Cc1cc(-n2ncc(=O)[nH]c2=O)ccc1C(=O)c1ccccc1Cl  Unknown   
1    CHEMBL6328   Cc1cc(-n2ncc(=O)[nH]c2=O)ccc1C(=O)c1ccc(C#N)cc1  Unknown   
2  CHEMBL265667  Cc1cc(-n2ncc(=O)[nH]c2=O)cc(C)c1C(O)c1ccc(Cl)cc1  Unknown   
3    CHEMBL6362      Cc1ccc(C(=O)c2ccc(-n3ncc(=O)[nH]c3=O)cc2)cc1  Unknown   
4  CHEMBL267864    Cc1cc(-n2ncc(=O)[nH]c2=O)ccc1C(=O)c1ccc(Cl)cc1  Unknown   

   Molecular_Weight     LogP  H_Bond_Donors  H_Bond_Acceptors  \
0           341.754  2.11362              1                 5   
1           332.319  1.33190              1                 6   
2           357.797  2.27274              2                 5   
3           307.309  1.46022              1                 5   
4           341.754  2.11362              1                 5   

   Rotatable_Bonds  Passes_Lipinski  
0                3             True  
1                3             True 

In [5]:
# 14.5 ADMET Properties

# Fix: SAScore is not in Descriptors module, need alternative method to calculate synthetic accessibility
def calculate_synthetic_accessibility(mol):
    """
    Calculate synthetic accessibility score as an alternative method
    Args:
        mol (rdkit.Chem.Mol): RDKit molecule object
    Returns:
        float: Synthetic accessibility score (lower values = easier to synthesize)
    """
    # Simple alternative implementation: estimation based on molecular complexity
    # In real applications, other RDKit functions or external libraries can be used
    try:
        # Use branching factor as a simple alternative
        if mol.GetNumAtoms() > 0:
            # Simplified SAScore calculation
            return min(10.0, mol.GetNumAtoms() / 10.0)  # Simple linear relationship
        else:
            return 0.0
    except:
        return 0.0

# Calculate ADMET-related properties
def calculate_admet_properties(smiles):
    """
    Calculate ADMET (Absorption, Distribution, Metabolism, Excretion, Toxicity) properties
    Args:
        smiles (str): SMILES representation of the molecule
    Returns:
        dict: Dictionary containing ADMET properties or None if invalid SMILES
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    return {
        'TPSA': Descriptors.TPSA(mol),  # Topological Polar Surface Area
        'Num_Rings': Descriptors.RingCount(mol),
        'Fraction_SP3': Descriptors.FractionCSP3(mol),
        # Fix: Use alternative method to calculate synthetic accessibility
        'SAScore': calculate_synthetic_accessibility(mol)
    }

# Add ADMET properties to dataframe
her2_df['admet'] = her2_df['smiles'].apply(calculate_admet_properties)
admet_df = her2_df['admet'].apply(pd.Series)
her2_df = pd.concat([her2_df.drop('admet', axis=1), admet_df], axis=1)

print("ADMET properties calculated:")
print(her2_df.head())

# Plot TPSA vs LogP
if 'LogP' in her2_df.columns and 'TPSA' in her2_df.columns:
    plt.figure(figsize=(10, 6))
    plt.scatter(her2_df['LogP'], her2_df['TPSA'], alpha=0.7)
    plt.xlabel('LogP')
    plt.ylabel('TPSA (Å²)')
    plt.title('TPSA vs LogP for HER2 Inhibitors')
    plt.grid(True)
    # Use savefig instead of show()
    plt.savefig('tpsa_vs_logp.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("TPSA vs LogP plot saved as 'tpsa_vs_logp.png'")

ADMET properties calculated:
      chembl_id                                            smiles     name  \
0    CHEMBL6329      Cc1cc(-n2ncc(=O)[nH]c2=O)ccc1C(=O)c1ccccc1Cl  Unknown   
1    CHEMBL6328   Cc1cc(-n2ncc(=O)[nH]c2=O)ccc1C(=O)c1ccc(C#N)cc1  Unknown   
2  CHEMBL265667  Cc1cc(-n2ncc(=O)[nH]c2=O)cc(C)c1C(O)c1ccc(Cl)cc1  Unknown   
3    CHEMBL6362      Cc1ccc(C(=O)c2ccc(-n3ncc(=O)[nH]c3=O)cc2)cc1  Unknown   
4  CHEMBL267864    Cc1cc(-n2ncc(=O)[nH]c2=O)ccc1C(=O)c1ccc(Cl)cc1  Unknown   

   Molecular_Weight     LogP  H_Bond_Donors  H_Bond_Acceptors  \
0           341.754  2.11362              1                 5   
1           332.319  1.33190              1                 6   
2           357.797  2.27274              2                 5   
3           307.309  1.46022              1                 5   
4           341.754  2.11362              1                 5   

   Rotatable_Bonds  Passes_Lipinski    TPSA  Num_Rings  Fraction_SP3  SAScore  
0                3             

In [6]:
# 14.6 Synthetic Accessibility

# Analyze synthetic accessibility
if 'SAScore' in her2_df.columns:
    plt.figure(figsize=(10, 6))
    sns.histplot(her2_df['SAScore'].dropna(), bins=20, kde=True)
    plt.title('Synthetic Accessibility Score Distribution')
    plt.xlabel('SAScore (Lower = Easier to Synthesize)')
    # Use savefig instead of show()
    plt.savefig('synthetic_accessibility.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("Synthetic accessibility plot saved as 'synthetic_accessibility.png'")

    # Show molecules with best synthetic accessibility
    easy_to_synthesize = her2_df.nsmallest(5, 'SAScore')
    print("Molecules with best synthetic accessibility:")
    for _, row in easy_to_synthesize.iterrows():
        print(f"{row['name']} (SAScore: {row['SAScore']:.2f})")


Synthetic accessibility plot saved as 'synthetic_accessibility.png'
Molecules with best synthetic accessibility:
BROMOENOL LACTONE (SAScore: 1.90)
Unknown (SAScore: 1.90)
Unknown (SAScore: 1.90)
Unknown (SAScore: 2.00)
Unknown (SAScore: 2.20)


In [7]:
# 14.7 Pharmacophore

# Generate and visualize pharmacophore features
def generate_pharmacophore(smiles):
    """
    Generate 2D pharmacophore features for a molecule
    Args:
        smiles (str): SMILES representation of the molecule
    Returns:
        fingerprint: Pharmacophore fingerprint or None if invalid SMILES
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    # Generate 2D pharmacophore
    pharmacophore = Generate.Gen2DFingerprint(mol, Gobbi_Pharm2D.factory)
    return pharmacophore

# Add pharmacophore to dataframe
her2_df['pharmacophore'] = her2_df['smiles'].apply(generate_pharmacophore)

# Visualize a molecule with its pharmacophore features
def visualize_molecule_with_features(smiles):
    """
    Visualize molecule structure
    Args:
        smiles (str): SMILES representation of the molecule
    Returns:
        PIL.Image: Image object or None if invalid SMILES
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        # Save molecule image to file instead of displaying
        img = Draw.MolToImage(mol, highlightAtoms=range(mol.GetNumAtoms()), size=(300, 300))
        return img
    return None

# Visualize first molecule
if not her2_df.empty:
    img = visualize_molecule_with_features(her2_df['smiles'].iloc[0])
    if img:
        img.save("molecule_with_features.png")
        print("Molecule visualization saved as 'molecule_with_features.png'")


Molecule visualization saved as 'molecule_with_features.png'


In [8]:
# 14.8 Quantitative Structure-Activity Relationship

# Generate molecular fingerprints for QSAR
def generate_fingerprints(smiles_list, radius=2, n_bits=1024):
    """
    Generate Morgan fingerprints for a list of SMILES
    Args:
        smiles_list (list): List of SMILES strings
        radius (int): Radius for Morgan fingerprint generation
        n_bits (int): Number of bits for fingerprint
    Returns:
        list: List of fingerprints (None for invalid SMILES)
    """
    fingerprints = []
    for smiles in smiles_list:
        mol = Chem.MolFromSmiles(smiles)
        if mol is not None:
            fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
            fingerprints.append(fp)
        else:
            fingerprints.append(None)
    return fingerprints

# Generate fingerprints (using a subset for demonstration)
sample_df = her2_df.dropna().head(30)  # Use a subset for demonstration
if not sample_df.empty:
    fingerprints = generate_fingerprints(sample_df['smiles'].tolist())

    # Create a simple QSAR model (using synthetic activity data for demonstration)
    # In a real scenario, you would have experimental activity data
    np.random.seed(42)
    sample_df['pIC50'] = np.random.uniform(5, 9, len(sample_df))  # Synthetic activity data

    # Prepare features and target
    # Filter out None values
    valid_data = [(fp, y_val) for fp, y_val in zip(fingerprints, sample_df['pIC50'].values) if fp is not None]
    if valid_data:
        X, y = zip(*valid_data)
        X = list(X)
        y = list(y)

        # Split data
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        # Train Random Forest model
        model = RandomForestRegressor(n_estimators=100, random_state=42)
        model.fit(X_train, y_train)

        # Make predictions
        y_pred = model.predict(X_test)

        # Evaluate model
        mse = mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        print(f"Model Performance: MSE = {mse:.3f}, R² = {r2:.3f}")

        # Plot predicted vs actual
        plt.figure(figsize=(8, 6))
        plt.scatter(y_test, y_pred, alpha=0.7)
        plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], 'r--')
        plt.xlabel('Actual pIC50')
        plt.ylabel('Predicted pIC50')
        plt.title('QSAR Model: Predicted vs Actual pIC50')
        # Use savefig instead of show()
        plt.savefig('qsar_model.png', dpi=300, bbox_inches='tight')
        plt.close()
        print("QSAR model plot saved as 'qsar_model.png'")


[16:24:08] DEPRECATION WARNING: please use MorganGenerator
[16:24:08] DEPRECATION WARNING: please use MorganGenerator
[16:24:08] DEPRECATION WARNING: please use MorganGenerator
[16:24:08] DEPRECATION WARNING: please use MorganGenerator
[16:24:08] DEPRECATION WARNING: please use MorganGenerator
[16:24:08] DEPRECATION WARNING: please use MorganGenerator
[16:24:08] DEPRECATION WARNING: please use MorganGenerator
[16:24:08] DEPRECATION WARNING: please use MorganGenerator
[16:24:08] DEPRECATION WARNING: please use MorganGenerator
[16:24:08] DEPRECATION WARNING: please use MorganGenerator
[16:24:08] DEPRECATION WARNING: please use MorganGenerator
[16:24:08] DEPRECATION WARNING: please use MorganGenerator
[16:24:08] DEPRECATION WARNING: please use MorganGenerator
[16:24:08] DEPRECATION WARNING: please use MorganGenerator
[16:24:08] DEPRECATION WARNING: please use MorganGenerator
[16:24:08] DEPRECATION WARNING: please use MorganGenerator
[16:24:08] DEPRECATION WARNING: please use MorganGenerat

Model Performance: MSE = 2.695, R² = -1.109
QSAR model plot saved as 'qsar_model.png'


In [9]:
# 14.9 Drug-Target Interaction

# Prepare ligand for docking (using a sample molecule)
if not her2_df.empty:
    sample_smiles = her2_df['smiles'].iloc[0]
    mol = Chem.MolFromSmiles(sample_smiles)

    if mol is not None:
        # Add hydrogens and generate 3D structure
        mol = Chem.AddHs(mol)
        AllChem.EmbedMolecule(mol, randomSeed=42)
        AllChem.MMFFOptimizeMolecule(mol)

        # Save ligand as PDB
        Chem.MolToPDBFile(mol, "ligand.pdb")
        print("Ligand prepared and saved as ligand.pdb")

    # Simple visualization of protein-ligand complex
    def visualize_complex(protein_file, ligand_file):
        """
        Visualize protein-ligand complex
        Args:
            protein_file (str): Path to protein PDB file
            ligand_file (str): Path to ligand PDB file
        Returns:
            py3Dmol.view: Visualization object or None if failed
        """
        if not os.path.exists(protein_file) or not os.path.exists(ligand_file):
            print("Protein or ligand file not found")
            return None

        try:
            with open(protein_file, "r") as f:
                protein_data = f.read()

            with open(ligand_file, "r") as f:
                ligand_data = f.read()

            view = py3Dmol.view(width=400, height=300)
            view.addModel(protein_data, "pdb")
            view.addModel(ligand_data, "pdb")
            view.setStyle({'model': -1}, {'stick': {'color': 'green'}})
            view.setStyle({'model': 0}, {'cartoon': {'color': 'spectrum'}})
            view.zoomTo()
            return view
        except Exception as e:
            print(f"Error visualizing complex: {e}")
            return None

    if os.path.exists("3PP0.pdb") and os.path.exists("ligand.pdb"):
        view = visualize_complex("3PP0.pdb", "ligand.pdb")
        if view:
            with open("complex_view.html", "w") as f:
                f.write(view._make_html())
            print("Protein-ligand complex visualization saved as 'complex_view.html'")


Ligand prepared and saved as ligand.pdb
Protein-ligand complex visualization saved as 'complex_view.html'


In [10]:
# 14.10 Virtual Screening

# Simple virtual screening using similarity search
def similarity_screening(query_smiles, screening_library, top_n=5):
    """
    Perform virtual screening using Tanimoto similarity
    Args:
        query_smiles (str): SMILES of query molecule
        screening_library (list): List of SMILES to screen against
        top_n (int): Number of top hits to return
    Returns:
        pandas.DataFrame: DataFrame with top similar compounds
    """
    query_mol = Chem.MolFromSmiles(query_smiles)
    if query_mol is None:
        print("Invalid query SMILES")
        return pd.DataFrame()

    query_fp = AllChem.GetMorganFingerprintAsBitVect(query_mol, 2)

    similarities = []
    for smiles in screening_library:
        mol = Chem.MolFromSmiles(smiles)
        if mol:
            fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2)
            similarity = DataStructs.TanimotoSimilarity(query_fp, fp)
            similarities.append(similarity)
        else:
            similarities.append(0)

    # Get top N similar compounds
    screening_df = pd.DataFrame({'smiles': screening_library, 'similarity': similarities})
    return screening_df.nlargest(top_n, 'similarity')

# Perform screening with a known active compound as query
if not her2_df.empty and 'smiles' in her2_df.columns:
    known_active = her2_df['smiles'].iloc[0]  # Using first compound as known active
    screening_library = her2_df['smiles'].tolist()  # Using our HER2 inhibitors as library

    top_hits = similarity_screening(known_active, screening_library)
    print("Top hits from virtual screening:")
    print(top_hits)

Top hits from virtual screening:
                                              smiles  similarity
0       Cc1cc(-n2ncc(=O)[nH]c2=O)ccc1C(=O)c1ccccc1Cl    1.000000
5         Cc1cc(-n2ncc(=O)[nH]c2=O)ccc1C(=O)c1ccccc1    0.740000
6   Cc1cc(Br)ccc1C(=O)c1ccc(-n2ncc(=O)[nH]c2=O)cc1Cl    0.735849
4     Cc1cc(-n2ncc(=O)[nH]c2=O)ccc1C(=O)c1ccc(Cl)cc1    0.725490
7  O=C(c1ccc(Cl)cc1Cl)c1ccc(-n2ncc(=O)[nH]c2=O)cc1Cl    0.666667


[16:26:09] DEPRECATION WARNING: please use MorganGenerator
[16:26:09] DEPRECATION WARNING: please use MorganGenerator
[16:26:09] DEPRECATION WARNING: please use MorganGenerator
[16:26:09] DEPRECATION WARNING: please use MorganGenerator
[16:26:09] DEPRECATION WARNING: please use MorganGenerator
[16:26:09] DEPRECATION WARNING: please use MorganGenerator
[16:26:09] DEPRECATION WARNING: please use MorganGenerator
[16:26:09] DEPRECATION WARNING: please use MorganGenerator
[16:26:09] DEPRECATION WARNING: please use MorganGenerator
[16:26:09] DEPRECATION WARNING: please use MorganGenerator
[16:26:09] DEPRECATION WARNING: please use MorganGenerator
[16:26:09] DEPRECATION WARNING: please use MorganGenerator
[16:26:09] DEPRECATION WARNING: please use MorganGenerator
[16:26:09] DEPRECATION WARNING: please use MorganGenerator
[16:26:09] DEPRECATION WARNING: please use MorganGenerator
[16:26:09] DEPRECATION WARNING: please use MorganGenerator
[16:26:09] DEPRECATION WARNING: please use MorganGenerat